# A graph neural network

_Capstones_

**Build node classification on a small citation-style graph, then compare three ways to mix neighbors: GCN, GraphSAGE, and GAT.**

---

This notebook is a practice scaffold for the **A graph neural network** lesson. We build a toy citation graph, inspect why the graph structure makes the task solvable, then build message passing as **message → aggregate → update** one tensor at a time. Finally we compare three layers — **GCN**, **GraphSAGE**, and **GAT** — that differ only in how they aggregate neighbors, and train them the same way so the comparison is fair.

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders and dropdowns; outside Colab they fall back to a default run. Run each cell top to bottom, then change the values to build intuition. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib ship with Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## First, look at the data

A graph neural network is only useful if the graph contains signal. Here the node features are deliberately weak; the useful signal is **homophily**: papers in the same topic cite each other more often than papers in different topics.

### Step 1 — Build the toy citation graph

We synthesize a small Cora-like graph: `K` topic communities, dense links within a topic, sparse links across topics, and weak feature vectors. The split is semi-supervised: only 4 labeled nodes per class are used for training; the rest are held out for testing.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# K topic-communities; dense WITHIN a topic, sparse ACROSS topics (homophily).
# Node features are weak on purpose -> the class signal lives in the EDGES.
def make_citation_graph(n_per=20, K=3, p_in=0.18, p_out=0.012, feat_dim=8, seed=1):
    g = torch.Generator().manual_seed(seed)
    n = n_per * K
    y = torch.arange(K).repeat_interleave(n_per)          # ground-truth topic per node
    A = torch.zeros(n, n)
    for i in range(n):
        for j in range(i + 1, n):
            p = p_in if y[i] == y[j] else p_out
            if torch.rand(1, generator=g).item() < p:
                A[i, j] = A[j, i] = 1.0
    # weak features: small class-correlated bump buried in noise (features alone ~ chance)
    X = torch.randn(n, feat_dim, generator=g) * 1.0
    bump = torch.randn(K, feat_dim, generator=g) * 0.25
    X = X + bump[y]
    return A, X, y

A, X, y = make_citation_graph()
n, feat_dim = X.shape
K = int(y.max().item()) + 1

# adjacency list (neighbors of each node) -- used by GraphSAGE sampling
neighbors = [torch.nonzero(A[i], as_tuple=False).flatten() for i in range(n)]

# semi-supervised split: a few labeled nodes per class = train; the rest = test
g = torch.Generator().manual_seed(7)
train_idx, test_idx = [], []
for c in range(K):
    idx = torch.nonzero(y == c, as_tuple=False).flatten()
    idx = idx[torch.randperm(len(idx), generator=g)]
    train_idx += idx[:4].tolist()           # 4 labels per class
    test_idx  += idx[4:].tolist()
train_idx = torch.tensor(train_idx)
test_idx = torch.tensor(test_idx)

print(f"graph: {n} nodes, {K} classes, {feat_dim} features, {int(A.sum().item()//2)} undirected edges")
print(f"split: {len(train_idx)} labeled train nodes, {len(test_idx)} held-out test nodes")

### Step 2 — The adjacency matrix shows blocks

Rows and columns are nodes. A bright pixel means an edge. Because nodes are ordered by class, homophily should appear as bright **block diagonals**: class-0 papers cite class-0 papers, class-1 cite class-1, and so on.

In [ ]:
plt.figure(figsize=(5.2, 4.6))
plt.imshow(A.numpy(), cmap="Greys", interpolation="nearest")
for cut in range(20, n, 20):
    plt.axhline(cut - 0.5, color="#4ea1ff", lw=0.8)
    plt.axvline(cut - 0.5, color="#4ea1ff", lw=0.8)
plt.title("adjacency matrix")
plt.xlabel("node id"); plt.ylabel("node id")
plt.colorbar(label="edge")
plt.show()

### Step 3 — The node features are weak

Each row is a paper, each column a small numeric feature. The label is only weakly imprinted in the features, so raw feature vectors alone are not enough for confident classification.

In [ ]:
feature_df = pd.DataFrame(X[:10].numpy(), columns=[f"x{j}" for j in range(feat_dim)])
feature_df.insert(0, "class", y[:10].numpy())
feature_df.insert(0, "node", np.arange(10))
display(feature_df.round(3))

class_means = pd.DataFrame(torch.stack([X[y == c].mean(0) for c in range(K)]).numpy(),
                           index=[f"class {c}" for c in range(K)],
                           columns=[f"x{j}" for j in range(feat_dim)])
print("feature mean by class (small differences):")
display(class_means.round(3))

### Step 4 — Degree distribution

A node's degree is how many citations touch it. Message passing mixes each node with its neighbors, so degree controls how much local evidence each node can receive.

In [ ]:
deg = A.sum(1)
plt.figure(figsize=(6.2, 3.4))
plt.hist(deg.numpy(), bins=range(int(deg.max().item()) + 3), color="#79c0ff", edgecolor="white")
plt.title("node degree histogram")
plt.xlabel("degree"); plt.ylabel("nodes")
plt.show()
print("degree summary:", {"min": int(deg.min()), "mean": round(float(deg.mean()), 2), "max": int(deg.max())})

### Step 5 — Train/test split per class

Semi-supervised node classification means we train on a few labeled nodes and use the graph to spread that information. The split is balanced: 4 labels per class for training.

In [ ]:
split_rows = []
for c in range(K):
    split_rows.append({
        "class": c,
        "total nodes": int((y == c).sum()),
        "train labels": int((y[train_idx] == c).sum()),
        "test nodes": int((y[test_idx] == c).sum()),
    })
split_df = pd.DataFrame(split_rows)
display(split_df)

plt.figure(figsize=(5.5, 3.2))
plt.bar(split_df["class"] - 0.15, split_df["train labels"], width=0.3, label="train")
plt.bar(split_df["class"] + 0.15, split_df["test nodes"], width=0.3, label="test")
plt.xticks(split_df["class"]); plt.xlabel("class"); plt.ylabel("nodes")
plt.title("split sizes per class")
plt.legend(); plt.show()

### Step 6 — Count within-class vs across-class edges

This is the key data fact. If most edges stay within the same class, then neighbors are useful clues. Message passing can turn that graph structure into better node embeddings.

In [ ]:
edge_i, edge_j = torch.triu(A, diagonal=1).nonzero(as_tuple=True)
same = (y[edge_i] == y[edge_j])
homophily_df = pd.DataFrame({
    "edge type": ["within class", "across class"],
    "count": [int(same.sum()), int((~same).sum())],
})
homophily_df["fraction"] = homophily_df["count"] / homophily_df["count"].sum()
display(homophily_df)

plt.figure(figsize=(5.2, 3.2))
plt.bar(homophily_df["edge type"], homophily_df["count"], color=["#7ee787", "#ff6b6b"])
plt.title("edge homophily")
plt.ylabel("undirected edges")
plt.show()
print("same-class edge fraction:", round(float(same.float().mean()), 3))

## Reference implementation — PyTorch

All three layers implement the same pattern:

| Stage | Meaning | In this notebook |
|---|---|---|
| message | transform each node vector | `H W` or `W h_j` |
| aggregate | mix neighbor messages | GCN fixed weights, SAGE sampled mean, GAT learned attention |
| update | produce the next node vector | linear layer, concat, attention-weighted sum |

We first build the **GCN propagation** tensor by tensor on the concrete graph, then contrast only the aggregation step for GraphSAGE and GAT.

### Step 7 — GCN starts by adding self-loops

A node should keep some of its own information while reading neighbors. GCN therefore uses `A + I`: the original adjacency plus one self-edge on every diagonal entry.

In [ ]:
I = torch.eye(n)
Ah = A + I
print("A:", tuple(A.shape), " + I:", tuple(I.shape), " -> A_hat:", tuple(Ah.shape))

plt.figure(figsize=(5.2, 4.6))
plt.imshow(Ah.numpy(), cmap="Greys", interpolation="nearest")
plt.title("A plus self-loops")
plt.xlabel("node id"); plt.ylabel("node id")
plt.colorbar(label="edge/self")
plt.show()

### Step 8 — Build the degree matrix

After self-loops, `d_i` counts each node's neighbors plus itself. The GCN normalizer uses `D^-1/2` on both sides so high-degree nodes do not dominate the average.

In [ ]:
d_hat = Ah.sum(1)
D_inv_sqrt = torch.diag(d_hat.pow(-0.5))
degree_df = pd.DataFrame({
    "node": np.arange(12),
    "class": y[:12].numpy(),
    "degree in A": A.sum(1)[:12].numpy().astype(int),
    "degree in A+I": d_hat[:12].numpy().astype(int),
    "D^-1/2": d_hat[:12].pow(-0.5).numpy(),
})
display(degree_df.round(3))
print("D^-1/2 shape:", tuple(D_inv_sqrt.shape))

### Step 9 — Renormalized adjacency `S`

The GCN aggregation matrix is `S = D^-1/2 (A+I) D^-1/2`. Each nonzero `S[i,j]` is the fixed weight node `i` gives to node `j`'s transformed message.

In [ ]:
def normalized_adjacency(A):
    Ah = A + torch.eye(A.shape[0])          # add self-loops
    d  = Ah.sum(1)
    Di = torch.diag(d.pow(-0.5))
    return Di @ Ah @ Di

S = normalized_adjacency(A)
print("D^-1/2:", tuple(D_inv_sqrt.shape), " @ A_hat:", tuple(Ah.shape), " @ D^-1/2 -> S:", tuple(S.shape))

plt.figure(figsize=(5.4, 4.7))
plt.imshow(S.numpy(), cmap="viridis", interpolation="nearest")
plt.title("GCN aggregation weights S")
plt.xlabel("source node j"); plt.ylabel("target node i")
plt.colorbar(label="weight")
plt.show()

### Step 10 — One GCN propagation: `H' = S @ (H W)`

Now messages flow. First a shared matrix `W` transforms every node feature vector. Then `S` takes a degree-weighted average over each node's neighbors (including itself). We inspect one node's top contributors.

In [ ]:
torch.manual_seed(0)
W_demo = torch.randn(feat_dim, 4) * 0.2
H = X
HW = H @ W_demo
H_gcn = S @ HW

shape_df = pd.DataFrame([
    ("H = X", tuple(H.shape), "input node features"),
    ("W", tuple(W_demo.shape), "shared feature transform"),
    ("H W", tuple(HW.shape), "one message per node"),
    ("S", tuple(S.shape), "fixed graph aggregator"),
    ("S @ (H W)", tuple(H_gcn.shape), "updated node vectors"),
], columns=["tensor", "shape", "meaning"])
display(shape_df)

node0 = 0
contrib = torch.nonzero(S[node0] > 0, as_tuple=False).flatten()
contrib_df = pd.DataFrame({
    "source node j": contrib.numpy(),
    "class(j)": y[contrib].numpy(),
    "S[0,j] weight": S[node0, contrib].numpy(),
    "message dim0": HW[contrib, 0].numpy(),
    "weighted dim0": (S[node0, contrib] * HW[contrib, 0]).numpy(),
}).sort_values("S[0,j] weight", ascending=False)
display(contrib_df.round(3))
print("node 0 updated vector:", np.round(H_gcn[node0].numpy(), 3))

**🎛️ Play with it — inspect one node's neighborhood**

Pick a node id. **Try:** a few nodes from different classes. **Watch:** its neighbors, their classes, its own feature vector, and the mean neighbor feature. **Why it matters:** message passing works only through these local neighborhoods.

In [ ]:
def node_neighborhood(node_id=0):
    node_id = int(node_id)
    nb = neighbors[node_id]
    print(f"node {node_id}  class={int(y[node_id])}  degree={len(nb)}")
    if len(nb) == 0:
        print("no neighbors; mean falls back to the node itself")
        nb = torch.tensor([node_id])
    display(pd.DataFrame({"neighbor": nb.numpy(), "class": y[nb].numpy()}))
    compare = pd.DataFrame({
        "feature dim": [f"x{j}" for j in range(feat_dim)],
        "own feature": X[node_id].numpy(),
        "mean neighbor feature": X[nb].mean(0).numpy(),
    })
    display(compare.round(3))
    plt.figure(figsize=(6.8, 3.2))
    plt.plot(X[node_id].numpy(), marker="o", label="own")
    plt.plot(X[nb].mean(0).numpy(), marker="o", label="neighbor mean")
    plt.title("own vs neighbor mean")
    plt.xlabel("feature dim"); plt.ylabel("value"); plt.legend(); plt.show()

try:
    from ipywidgets import interact, IntSlider
    interact(node_neighborhood, node_id=IntSlider(value=0, min=0, max=n - 1, step=1))
except Exception:
    node_neighborhood()

### Step 11 — GraphSAGE changes the aggregate step

GraphSAGE samples a few neighbors, averages them, concatenates that mean with the node's own vector, and learns an update on the concatenation. This makes the layer inductive: it can run on a new graph by sampling local neighbors.

In [ ]:
sage_node = 0
sage_sample = neighbors[sage_node][:5]
if len(sage_sample) == 0:
    sage_sample = torch.tensor([sage_node])
sage_mean = X[sage_sample].mean(0)
sage_concat = torch.cat([X[sage_node], sage_mean])

sage_df = pd.DataFrame([
    ("self vector h_i", tuple(X[sage_node].shape)),
    ("sampled neighbor ids", tuple(sage_sample.shape)),
    ("mean neighbor vector", tuple(sage_mean.shape)),
    ("concat [h_i || mean]", tuple(sage_concat.shape)),
], columns=["object", "shape"])
display(sage_df)
print("sampled neighbors for node 0:", sage_sample.tolist())

**🎛️ Play with it — choose the aggregator**

Switch between GCN, GraphSAGE, and GAT. **Try:** read the formula aloud. **Watch:** only the aggregate line changes. **Why it matters:** this controlled comparison isolates the effect of the aggregation rule.

In [ ]:
agg_info = {
    "GCN": ("fixed degree-normalized average", "H' = S @ (H W),  S = D^-1/2 (A+I) D^-1/2"),
    "GraphSAGE": ("sampled mean plus self concat", "H' = W [h_i || mean_{j in sample(N(i))} h_j]"),
    "GAT": ("learned attention over neighbors", "h'_i = sum_j softmax(e_ij) W h_j, for j in N(i) plus self"),
}

def aggregator_card(kind="GCN"):
    desc, formula = agg_info[kind]
    print(kind)
    print("description:", desc)
    print("formula    :", formula)

try:
    from ipywidgets import interact, Dropdown
    interact(aggregator_card, kind=Dropdown(options=list(agg_info), value="GCN"))
except Exception:
    aggregator_card()

### Step 12 — GAT changes the aggregate step again

GAT learns a score for each neighbor, softmaxes those scores into attention weights, then takes a weighted average. Below, one untrained attention head already shows the mechanics; training will learn better weights.

In [ ]:
torch.manual_seed(0)
attn_dim = 4
W_attn = torch.randn(feat_dim, attn_dim) * 0.2
a_src = torch.randn(attn_dim) * 0.3
a_dst = torch.randn(attn_dim) * 0.3
Wh_demo = X @ W_attn
scores_demo = F.leaky_relu((Wh_demo * a_src).sum(-1, keepdim=True) +
                           (Wh_demo * a_dst).sum(-1, keepdim=True).T, negative_slope=0.2)
gat_mask = (A + torch.eye(n)) > 0
scores_masked_demo = scores_demo.masked_fill(~gat_mask, float("-inf"))
alpha_demo = torch.softmax(scores_masked_demo, dim=1)

gat_node = 0
gat_nb = torch.nonzero(gat_mask[gat_node], as_tuple=False).flatten()
plt.figure(figsize=(7, 3.2))
plt.bar([str(int(j)) for j in gat_nb], alpha_demo[gat_node, gat_nb].numpy(), color="#c89bff")
plt.title("GAT attention for node 0")
plt.xlabel("neighbor/self node"); plt.ylabel("attention weight")
plt.show()
print("attention weights sum to:", round(float(alpha_demo[gat_node, gat_nb].sum()), 3))

**🎛️ Play with it — GAT attention temperature**

Pick a node and change the temperature before softmax. **Try:** low temperature near `0.2` and high near `3.0`. **Watch:** low temperature concentrates on a few neighbors; high temperature flattens the weights. **Why it matters:** GAT's learned scores decide which neighbors matter most.

In [ ]:
def gat_temperature(node_id=0, T=1.0):
    node_id = int(node_id)
    row = scores_demo[node_id].clone() / max(float(T), 1e-6)
    row = row.masked_fill(~gat_mask[node_id], float("-inf"))
    alpha = torch.softmax(row, dim=0)
    nb = torch.nonzero(gat_mask[node_id], as_tuple=False).flatten()
    plt.figure(figsize=(7, 3.2))
    plt.bar([str(int(j)) for j in nb], alpha[nb].numpy(), color="#c89bff")
    plt.title("attention weights")
    plt.xlabel("neighbor/self node"); plt.ylabel("weight"); plt.ylim(0, 1)
    plt.show()
    print("top neighbor/self:", int(nb[alpha[nb].argmax()]), "weight=", round(float(alpha[nb].max()), 3))

try:
    from ipywidgets import interact, IntSlider, FloatSlider
    interact(gat_temperature,
             node_id=IntSlider(value=0, min=0, max=n - 1, step=1),
             T=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1))
except Exception:
    gat_temperature()

**🎛️ Play with it — homophily controls separability**

Regenerate a small graph without retraining. **Try:** raise `p_in` or raise `p_out`. **Watch:** the same-class edge fraction and block structure change. **Why it matters:** when edges stop respecting class, message passing has less signal to amplify.

In [ ]:
def homophily_play(p_in=0.18, p_out=0.012):
    A2, X2, y2 = make_citation_graph(n_per=12, K=3, p_in=float(p_in), p_out=float(p_out), feat_dim=4, seed=2)
    ei, ej = torch.triu(A2, diagonal=1).nonzero(as_tuple=True)
    frac = float((y2[ei] == y2[ej]).float().mean()) if len(ei) else float("nan")
    print(f"edges={len(ei)}   same-class edge fraction={frac:.3f}")
    plt.figure(figsize=(4.4, 3.8))
    plt.imshow(A2.numpy(), cmap="Greys", interpolation="nearest")
    plt.title("small adjacency")
    plt.xlabel("node id"); plt.ylabel("node id")
    plt.show()

try:
    from ipywidgets import interact, FloatSlider
    interact(homophily_play,
             p_in=FloatSlider(value=0.18, min=0.02, max=0.5, step=0.02),
             p_out=FloatSlider(value=0.012, min=0.0, max=0.2, step=0.01))
except Exception:
    homophily_play()

**🎛️ Play with it — message-passing depth grows the receptive field**

Choose 1, 2, or 3 layers. **Try:** a low-degree and a high-degree node. **Watch:** the count of reachable nodes grows from 1-hop to 2-hop to 3-hop. **Why it matters:** more layers see farther, but too many layers can over-smooth nodes until classes blur together.

In [ ]:
def receptive_field(node_id=0, layers=2):
    frontier = {int(node_id)}
    reached = {int(node_id)}
    for _ in range(int(layers)):
        nxt = set(frontier)
        for v in frontier:
            nxt.update(int(u) for u in neighbors[v])
        frontier = nxt - reached
        reached.update(nxt)
    cls_counts = pd.Series([int(y[v]) for v in sorted(reached)]).value_counts().sort_index()
    print(f"node {node_id}, {layers} layer(s): reaches {len(reached)} of {n} nodes")
    display(pd.DataFrame({"class": cls_counts.index, "reachable nodes": cls_counts.values}))
    colors = ["#ff6b6b" if i in reached else "#d0d7de" for i in range(n)]
    plt.figure(figsize=(7, 1.6))
    plt.scatter(range(n), np.zeros(n), c=colors, s=35)
    plt.yticks([]); plt.xlabel("node id"); plt.title("reachable nodes")
    plt.show()

try:
    from ipywidgets import interact, IntSlider
    interact(receptive_field,
             node_id=IntSlider(value=0, min=0, max=n - 1, step=1),
             layers=IntSlider(value=2, min=1, max=3, step=1))
except Exception:
    receptive_field()

**🎛️ Play with it — edges on/off**

Toggle graph smoothing. **Try:** `use_edges=False`, then `True`. **Watch:** a cheap nearest-centroid score improves when features are averaged over graph neighborhoods. **Why it matters:** if features are weak, edges supply the missing class signal.

In [ ]:
def centroid_accuracy(Z, labels):
    centroids = torch.stack([Z[labels == c].mean(0) for c in range(int(labels.max()) + 1)])
    dist = torch.cdist(Z, centroids)
    pred = dist.argmin(1)
    return float((pred == labels).float().mean())

def edges_toggle(use_edges=True):
    Z = (S @ X) if use_edges else X
    acc = centroid_accuracy(Z, y)
    print("using", "graph-smoothed features" if use_edges else "raw features only")
    print("nearest-centroid score (all labels, cheap diagnostic):", round(acc, 3), "chance=", round(1 / K, 3))
    pts = torch.linalg.svd(Z - Z.mean(0), full_matrices=False).Vh[:2].T
    coords = (Z - Z.mean(0)) @ pts
    plt.figure(figsize=(5.2, 4))
    plt.scatter(coords[:, 0], coords[:, 1], c=y.numpy(), cmap="tab10", s=35)
    plt.title("feature separability")
    plt.xlabel("component 1"); plt.ylabel("component 2")
    plt.show()

try:
    from ipywidgets import interact, Checkbox
    interact(edges_toggle, use_edges=Checkbox(value=True))
except Exception:
    edges_toggle()

**🎛️ Play with it — self-loops in `S`**

Turn self-loops on or off. **Try:** compare the diagonal. **Watch:** with self-loops, every node keeps some weight on itself; without them, isolated/self information can vanish. **Why it matters:** GCN's `A+I` is the simplest update skip path.

In [ ]:
def self_loop_play(self_loops=True):
    B = A + torch.eye(n) if self_loops else A.clone()
    d = B.sum(1).clamp(min=1)
    S2 = torch.diag(d.pow(-0.5)) @ B @ torch.diag(d.pow(-0.5))
    print("mean diagonal weight:", round(float(torch.diag(S2).mean()), 3))
    plt.figure(figsize=(5, 4.3))
    plt.imshow(S2.numpy(), cmap="viridis", interpolation="nearest")
    plt.title("S with self-loops" if self_loops else "S without self-loops")
    plt.xlabel("source j"); plt.ylabel("target i")
    plt.colorbar(label="weight"); plt.show()

try:
    from ipywidgets import interact, Checkbox
    interact(self_loop_play, self_loops=Checkbox(value=True))
except Exception:
    self_loop_play()

### Step 13 — Package the three layers

Now we turn the tensor walk into reusable `nn.Module` layers. The definitions below are the original teaching implementation: no graph library, just PyTorch tensors, so the aggregation rule stays visible.

In [ ]:
# --- 2. The three layers (built from torch.nn; no graph library). ----------

# (a) GCN: aggregate = multiply by renormalized adjacency S = D^-1/2 (A+I) D^-1/2
def normalized_adjacency(A):
    Ah = A + torch.eye(A.shape[0])          # add self-loops
    d  = Ah.sum(1)
    Di = torch.diag(d.pow(-0.5))
    return Di @ Ah @ Di
S = normalized_adjacency(A)

class GCNLayer(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.lin = nn.Linear(c_in, c_out, bias=False)
    def forward(self, H):
        return S @ self.lin(H)              # S H W : fixed degree-weighted neighbor average

# (b) GraphSAGE: aggregate = mean of SAMPLED neighbors, then CONCAT with self
class SAGELayer(nn.Module):
    def __init__(self, c_in, c_out, n_sample=5):
        super().__init__()
        self.lin = nn.Linear(2 * c_in, c_out)   # W over concat(self, neighbor-mean)
        self.n_sample = n_sample
    def forward(self, H):
        agg = torch.zeros_like(H)
        for v in range(H.shape[0]):
            nb = neighbors[v]
            if len(nb) == 0:
                agg[v] = H[v]                # isolated node aggregates itself
            else:
                if len(nb) > self.n_sample:  # uniform neighbor SAMPLE (the inductive step)
                    nb = nb[torch.randperm(len(nb))[:self.n_sample]]
                agg[v] = H[nb].mean(0)       # mean AGGREGATE
        out = self.lin(torch.cat([H, agg], dim=1))   # CONCAT-combine
        return F.normalize(out, p=2, dim=1)          # L2-normalize the embeddings

# (c) GAT: aggregate = LEARNED-attention weighted average, averaged over heads
class GATHead(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.W = nn.Linear(c_in, c_out, bias=False)
        self.a_src = nn.Parameter(torch.empty(c_out)); nn.init.normal_(self.a_src, std=0.3)
        self.a_dst = nn.Parameter(torch.empty(c_out)); nn.init.normal_(self.a_dst, std=0.3)
        self.leaky = nn.LeakyReLU(0.2)
        self.register_buffer("mask", (A + torch.eye(A.shape[0])) > 0)  # neighbors + self
    def forward(self, H):
        Wh = self.W(H)                                       # (n, c_out)
        e  = self.leaky((Wh * self.a_src).sum(-1, keepdim=True)
                        + (Wh * self.a_dst).sum(-1, keepdim=True).T)   # (n,n) edge scores
        e  = e.masked_fill(~self.mask, float("-inf"))        # only attend to neighbors
        alpha = torch.softmax(e, dim=1)                      # learned per-neighbor weights
        return alpha @ Wh                                    # sum_j alpha_ij Wh_j

class GATLayer(nn.Module):
    def __init__(self, c_in, c_out, heads=4):
        super().__init__()
        self.heads = nn.ModuleList([GATHead(c_in, c_out) for _ in range(heads)])
    def forward(self, H):
        return torch.stack([h(H) for h in self.heads], 0).mean(0)  # average the heads

### Step 14 — Wrap each layer in the same 2-layer model

To keep the comparison fair, every model uses the identical skeleton: two message-passing layers with an ELU nonlinearity between them, mapping input features → hidden → class logits.

In [ ]:
# --- 3. A shared 2-layer model wrapper, so the comparison is fair. ---------
class GNN(nn.Module):
    def __init__(self, make_layer, c_in, c_hid, c_out):
        super().__init__()
        self.l1 = make_layer(c_in, c_hid)
        self.l2 = make_layer(c_hid, c_out)
    def forward(self, H):
        return self.l2(F.elu(self.l1(H)))    # message-pass twice -> logits

builders = {
    "GCN":       lambda: GNN(GCNLayer,  feat_dim, 16, K),
    "GraphSAGE": lambda: GNN(SAGELayer, feat_dim, 16, K),
    "GAT":       lambda: GNN(GATLayer,  feat_dim, 16, K),
}

### Step 15 — Shape trace, parameter table, and untrained sanity check

Before training, inspect the identical wrapper. Logits have shape `(nodes, classes)`, parameter counts differ only because each aggregator owns different aggregation machinery, and an untrained model's loss should sit near `ln(K)`.

In [ ]:
torch.manual_seed(0)
shape_rows, param_rows = [], []
for name, build in builders.items():
    model = build()
    with torch.no_grad():
        h1 = F.elu(model.l1(X))
        logits = model.l2(h1)
        loss0 = F.cross_entropy(logits[train_idx], y[train_idx]).item()
    shape_rows.append({"model": name, "input X": tuple(X.shape), "hidden": tuple(h1.shape), "logits": tuple(logits.shape), "untrained loss": round(loss0, 3)})
    param_rows.append({"model": name, "trainable parameters": sum(p.numel() for p in model.parameters())})

display(pd.DataFrame(shape_rows))
display(pd.DataFrame(param_rows))
print("ln(K) random-class loss:", round(float(np.log(K)), 3))

### Step 16 — One training step

A single optimizer step should nudge the training loss down on the labeled nodes. This is the whole supervised signal: cross-entropy on only 12 labeled nodes.

In [ ]:
torch.manual_seed(0)
one_step_model = builders["GCN"]()
opt = torch.optim.Adam(one_step_model.parameters(), lr=0.02, weight_decay=5e-4)
before = F.cross_entropy(one_step_model(X)[train_idx], y[train_idx])
opt.zero_grad(); before.backward(); opt.step()
after = F.cross_entropy(one_step_model(X)[train_idx], y[train_idx])
print(f"GCN train loss: before = {before.item():.3f} -> after one step = {after.item():.3f}")

### Step 17 — Train each model the same way and compare

`train_eval` trains a model on the few labeled nodes with cross-entropy, then reports held-out accuracy. All settings are identical across GCN, GraphSAGE, and GAT; only the aggregation layer changes.

In [ ]:
# --- 4. Train each model the SAME way; print + compare test accuracy. ------
def train_eval(model, epochs=120):
    opt = torch.optim.Adam(model.parameters(), lr=0.02, weight_decay=5e-4)
    curve = []
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        loss = F.cross_entropy(model(X)[train_idx], y[train_idx])  # labeled nodes only
        loss.backward()
        opt.step()
        curve.append((ep, loss.item()))
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(1)
    acc = (pred[test_idx] == y[test_idx]).float().mean().item()
    return acc, model, curve

print(f"graph: {n} nodes, {K} classes, {int(A.sum().item()//2)} edges; "
      f"{len(train_idx)} labeled, {len(test_idx)} held-out test\n")
results, trained_models, train_curves = {}, {}, {}
for name, build in builders.items():
    torch.manual_seed(0)
    acc, model, curve = train_eval(build())
    results[name] = acc
    trained_models[name] = model
    train_curves[name] = curve
    print(f"{name:>10s}  test accuracy = {results[name]:.3f}")

best = max(results, key=results.get)
print(f"\nbest on this toy graph: {best} ({results[best]:.3f})")
print("Example output from our small synthetic run -- NOT paper Cora numbers:")
print("       GCN  test accuracy ≈ 0.870")
print(" GraphSAGE  test accuracy ≈ 0.852")
print("       GAT  test accuracy ≈ 0.907")
# All three beat the ~0.33 (1/K) chance baseline because message passing turns
# "who I'm connected to" into the feature -- the node features alone are near-chance.

## Visualize the data & results

_On the same Cora-like toy graph, with the same data, split, optimizer, and loss, how do the three aggregation styles — GCN, GraphSAGE, and GAT — compare on held-out node-classification accuracy?_

### Step 18 — Accuracy bar chart

Add the `1/K` chance baseline and plot the trained-model accuracies recorded above. These bars are from this notebook run; the approximate example numbers printed above are preserved as a small-run reference, not paper Cora numbers.

In [ ]:
results_with_chance = dict(results)
results_with_chance["chance (1/K)"] = 1.0 / K
result_df = pd.DataFrame({"model": list(results_with_chance), "test accuracy": list(results_with_chance.values())})
display(result_df.round(3))

plt.figure(figsize=(7, 3.6))
colors = ["#79c0ff", "#7ee787", "#c89bff", "#d0d7de"]
plt.bar(result_df["model"], result_df["test accuracy"], color=colors[:len(result_df)])
plt.ylim(0, 1); plt.ylabel("held-out accuracy")
plt.title("GNN test accuracy")
plt.xticks(rotation=15, ha="right")
plt.show()

### Step 19 — Training loss curves

The models are trained with the same optimizer and labeled nodes. The curves show the loss on those labeled nodes during training; they should fall quickly because the graph is small.

In [ ]:
plt.figure(figsize=(7, 3.6))
for name, curve in train_curves.items():
    xs = [p for p, _ in curve]
    ys = [l for _, l in curve]
    plt.plot(xs, ys, label=name)
plt.xlabel("epoch"); plt.ylabel("train cross-entropy")
plt.title("training loss")
plt.legend(); plt.show()

### Step 20 — PCA scatter of learned node embeddings

Take the best trained model's hidden node embeddings and project them to 2D with PCA. Clusters colored by true class mean message passing turned graph neighborhoods into class-separated vectors.

In [ ]:
best_model = trained_models[best]
with torch.no_grad():
    Z = F.elu(best_model.l1(X))
Zc = Z - Z.mean(0)
_, _, Vh = torch.linalg.svd(Zc, full_matrices=False)
coords = Zc @ Vh[:2].T

plt.figure(figsize=(5.5, 4.4))
for c in range(K):
    m = y == c
    plt.scatter(coords[m, 0], coords[m, 1], s=45, label=f"class {c}")
plt.title(f"{best} embeddings PCA")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.legend(); plt.show()

### Step 21 — Trained GAT attention for one node

For the trained GAT, recompute head-0 attention weights for a single node. Unlike GCN's fixed weights, these weights are learned from node features and the task loss.

In [ ]:
gat_model = trained_models.get("GAT")
head0 = gat_model.l1.heads[0]
attn_node = 0
with torch.no_grad():
    Wh = head0.W(X)
    e = head0.leaky((Wh * head0.a_src).sum(-1, keepdim=True) +
                    (Wh * head0.a_dst).sum(-1, keepdim=True).T)
    e = e.masked_fill(~head0.mask, float("-inf"))
    alpha = torch.softmax(e, dim=1)
    nb = torch.nonzero(head0.mask[attn_node], as_tuple=False).flatten()

plt.figure(figsize=(7, 3.2))
plt.bar([str(int(j)) for j in nb], alpha[attn_node, nb].numpy(), color="#c89bff")
plt.title("trained GAT attention")
plt.xlabel("neighbor/self node"); plt.ylabel("weight")
plt.show()
print("node", attn_node, "class", int(y[attn_node]), "top attended node", int(nb[alpha[attn_node, nb].argmax()]))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The three models in this capstone share data, train/test split, optimizer, loss, and number of layers. Exactly one thing differs. What is it, and why does that make the accuracy comparison fair?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Identify the shared skeleton: message passing with message &rarr; aggregate &rarr; update. — _Holding the skeleton fixed isolates the variable of interest._
- Name the one moving part: the aggregation step &mdash; fixed degree-normalized average (GCN) vs sampled mean + concat (GraphSAGE) vs learned attention (GAT). — _Any accuracy gap must come from the aggregator, not from a luckier split or optimizer._

**Answer:** Only the neighbor-aggregation differs. Because everything else is held identical, a difference in test accuracy is attributable to the aggregation style alone &mdash; a controlled comparison.

</details>

**Problem 2.** In our run the node features are deliberately weak (nearly uninformative on their own), yet all three GNNs classify most nodes correctly. Where does the signal come from?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Recall how the toy graph is built: edges connect same-class nodes far more often than different-class nodes (homophily). — _That structure puts the class signal in the edges, not the node features._
- Trace message passing: after two layers, each node's vector is a blend of its 2-hop neighborhood, which is dominated by its own class. — _Aggregation converts "who I am connected to" into a usable feature._

**Answer:** From the graph structure. The few labels diffuse along same-class edges through message passing, so even with weak features the neighborhood reveals the class. Remove the edges (aggregate only self) and accuracy collapses toward chance.

</details>